In [1]:
!conda info


     active environment : /users/hzheng29/miniconda3/envs/galaxy-ssl-1
    active env location : /users/hzheng29/miniconda3/envs/galaxy-ssl-1
            shell level : 1
       user config file : /users/hzheng29/.condarc
 populated config files : /oscar/home/hzheng29/.condarc
          conda version : 23.7.4
    conda-build version : 3.26.1
         python version : 3.11.5.final.0
       virtual packages : __archspec=1=x86_64
                          __glibc=2.34=0
                          __linux=5.14.0=0
                          __unix=0=0
       base environment : /oscar/rt/9.6/25/spack/x86_64_v3/anaconda3-2023.09-0-aqbcryind6ewgctu7wijluakv5mo3lo5  (read only)
      conda av data dir : /oscar/rt/9.6/25/spack/x86_64_v3/anaconda3-2023.09-0-aqbcryind6ewgctu7wijluakv5mo3lo5/etc/conda
  conda av metadata url : None
           channel URLs : https://repo.anaconda.com/pkgs/main/linux-64
                          https://repo.anaconda.com/pkgs/main/noarch
                          http

In [2]:
!pip install trl
# !pip uninstall torchao -y
# !pip cache purge
!pip install torchao
!pip install peft

  Using cached torch-2.12.1-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)


Using cached typer-0.25.1-py3-none-any.whl (58 kB)
Using cached torch-2.12.1-cp310-cp310-manylinux_2_28_x86_64.whl (532.1 MB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.6.0
    Uninstalling fsspec-2026.6.0:
      Successfully uninstalled fsspec-2026.6.0
  Attempting uninstall: typer━━━━━━━━━━━━━━━━━━━ 0/3 [fsspec]
    Found existing installation: typer 0.26.8 0/3 [fsspec]
    Uninstalling typer-0.26.8:╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [typer]
      Successfully uninstalled typer-0.26.8━━━━━━━━━━━━━━━━━━━ 1/3 [typer]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [torch]^C
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [torch]
  Using cached torch-2.12.1-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (31 kB)


Using cached torch-2.12.1-cp310-cp310-manylinux_2_28_x86_64.whl (532.1 MB)
^C
ERROR: Operation cancelled by user


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from torch.optim import AdamW
from trl import SFTTrainer
import torch

In [ ]:
!hf auth login

In [ ]:
# torch.cuda.empty_cache()

In [7]:
print(torch.cuda.is_available())

False


In [6]:
BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # 'mps' if torch.backends.mps.is_available() else 'cpu'
OUT_DIR = './model_output'

training_prompts = [
    [
        {
            'role': 'user', 'content': 'What\'s my name?'
        },
        {
            'role':'assistant', 'content': 'Your name is'
        }
    ]
]

training_targets = [
    'Regina Zheng'
]

##################
# Run model directly
###################

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    # dtype=torch.bfloat16,
    device_map=DEVICE
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"]
)

model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
print(model.device)

cpu


In [10]:
'''
Example of full chat template given by tokenizer.apply_chat_template(prompts):
--------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Oct 2024

You are a smart AI assistant who speaks like a pirate.<|eot_id|><|start_header_id|>user<|end_header_id|>
Where does the sun rises?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Aye aye
--------------
'''

def generate_input_output_pair(prompts, target_responses):

    # Make chat templates
    chat_templates = tokenizer.apply_chat_template(prompts, continue_final_message=True, tokenize=False)
    # full_response_text = [completed chat templates with target response and eos token]
    full_response_text = [
        (chat_template + '' + target_response + tokenizer.eos_token)
        for chat_template, target_response in zip(chat_templates, target_responses)
    ]
    # Token full response text
    input_ids_tokenized = tokenizer(
        full_response_text,
        return_tensors='pt',
        add_special_tokens=False
    )['input_ids']


    # Token target labels
    full_response_target_text = [' ' + target_response + tokenizer.eos_token for target_response in target_responses]
    labels_tokenized = tokenizer(
        full_response_target_text,
        add_special_tokens=False,
        return_tensors='pt',
        padding='max_length',
        max_length=input_ids_tokenized.shape[1]
    )['input_ids']

    # Set padding takens in labels to -100 so they are ignored
    labels_tokenized_fixed = torch.where(labels_tokenized != tokenizer.pad_token_id, labels_tokenized, -100)
    # Set the last token in labels to pad_token_id (tokenizer.eros_token_id in this case)
    labels_tokenized_fixed[:, -1] = tokenizer.pad_token_id

    # Shift input_ids left and labels right to create attention mask
    input_ids_tokenzied_left_shifted = input_ids_tokenized[:, :-1]
    labels_tokenized_right_shifted = labels_tokenized_fixed[:, 1:]

    attention_mask = input_ids_tokenzied_left_shifted != tokenizer.pad_token_id

    return {
        'input_ids': input_ids_tokenzied_left_shifted,
        'attention_mask': attention_mask,
        'labels': labels_tokenized_right_shifted
    }

def calculate_loss(logits, labels):
    loss_fct = torch.nn.CrossEntropyLoss()
    loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
    return loss


data = generate_input_output_pair(prompts=training_prompts, target_responses=training_targets)
data['input_ids'] = data['input_ids']#.to(DEVICE)
data['labels'] = data['labels']#.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

#####
def test():
    test_tokenized = tokenizer.apply_chat_template(
        training_prompts[0],
        add_generation_prompt=False,
        tokenize=True,
        padding=True,
        return_tensors='pt'
    )

    print(test_tokenized)

    test_out = model.generate(
        test_tokenized['input_ids'],#.to(DEVICE),
        # attention_mask=test_tokenized['attention_mask'].to(DEVICE),
        max_new_tokens=50
    )
    test_out_decoded = tokenizer.decode(test_out[0], skip_special_tokens=True)
    print(test_out_decoded)
#####

print('Before training test')
test()

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Before training test
{'input_ids': tensor([[128000, 128006,    882, 128007,    271,   3923,    596,    856,    836,
             30, 128009, 128006,  78191, 128007,    271,   7927,    836,    374,
         128009]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


user

What's my name?assistant

Your name isassistant

I apologize, but I don't have any information about your name. I'm a large language model, I don't have personal knowledge or memories, and I don't have the ability to know your name unless you tell me.


In [ ]:
for _ in range(10):
    for prompt in training_prompts:
        out = model(input_ids=data['input_ids'].to(DEVICE))
        loss = calculate_loss(out.logits, data['labels'].to(DEVICE)).mean()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f'Loss: {loss.item()}')


test()

###############
###############
# Using pipeline and trainer
###############
###############

# ds = Dataset.from_dict(training_prompts)

# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     load_in_4bit=True,
#     device_map="auto"
# )

# lora_cfg = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     target_modules=["q_proj","v_proj"]
# )
# model = get_peft_model(model, lora_cfg)

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=ds,
#     max_seq_length=1024,
#     dataset_text_field="instruction",  # we’ll override collator
#     args=dict(
#        per_device_train_batch_size=2,
#        gradient_accumulation_steps=16,
#        num_train_epochs=1,
#        fp16=True,
#        output_dir=OUT_DIR,
#        logging_steps=50,
#        save_strategy="epoch"
#     ),
# )
# trainer.train()
# trainer.model.save_pretrained(OUT_DIR)
# tokenizer.save_pretrained(OUT_DIR)